## In this code, we first use finite difference to calculate derivative, using different mesh for integration of H, and different differentiation step h
## Then we use spline to calculate the derivatives to confirm 

In [1]:

import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline
 
_trapz = getattr(np, 'trapezoid', None) or np.trapz   # numpy 1.x / 2.x
 
# =============================================================================
# CONFIGURATION
# =============================================================================
 
# Located by root-finding C'(pi, kappa) on a converged scan over [385, 400].
# The older value 386.79 gives C'(pi) = +2.5e-7 rather than 0.
KAPPA_STAR = 387.0522
 
DKAPPA = 1.0          # FD step in kappa, for a
H_PHI  = 0.002        # FD step in phi, for C'(pi) and C'''(pi)
 
TN         = 100_000  # points on the limit cycle (Youngmin's TN)
N_PERIODS  = 20       # periods of backward integration for the adjoint
RTOL, ATOL = 1e-12, 1e-12
 
N_LIST = [100, 200, 400, 800, 1600, 3200, 6400]   # trapezoid mesh for H
 
# steps swept in Table 2 (at the finest N); spans the plateau and both edges
H_SWEEP = [0.5, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001, 5e-4, 2e-4, 1e-4]
 
BASE = dict(alpha=0.01, beta=0.5, gamma=1/7, eta=1/7, theta=0.01, rho=0.01)
 
 
# =============================================================================
# SYSTEM
# =============================================================================
 
def f(t, y, p):
    S, I, v = y
    return [p['alpha']*(1 - S - I) - p['beta']*S*I - p['eta']*v*S,
            -p['gamma']*I + p['beta']*S*I,
            -v + 1/(1 + np.exp(-p['kappa']*(p['beta']*I - p['rho']
                                            - p['theta']*(1 - 2*v))))]
 
 
_S, _I, _v = sp.symbols('S I v')
_al, _be, _ga, _et, _th, _ka, _rh = sp.symbols(
    'alpha beta gamma eta theta kappa rho')
 
_F = sp.Matrix([
    _al*(1 - _S - _I) - _be*_S*_I - _et*_v*_S,
    -_ga*_I + _be*_S*_I,
    -_v + 1/(1 + sp.exp(-_ka*(_be*_I - _rh - _th*(1 - 2*_v))))])
 
_DF_func = sp.lambdify((_S, _I, _v, _al, _be, _ga, _et, _th, _ka, _rh),
                       _F.jacobian(sp.Matrix([_S, _I, _v])), modules='numpy')
 
 
def DF(S, I, v, p):
    return np.array(_DF_func(S, I, v, p['alpha'], p['beta'], p['gamma'],
                             p['eta'], p['theta'], p['kappa'], p['rho']))
 
 
def make_periodic_spline(t, y, T):
    """Periodic cubic spline; swap this one function to test other splines."""
    yy = np.asarray(y, dtype=float).copy()
    yy[-1] = yy[0]                       # CubicSpline(bc_type='periodic') needs this
    cs = CubicSpline(t, yy, bc_type='periodic')
    return lambda tq: cs(np.mod(tq, T))
 
 
# =============================================================================
# LIMIT CYCLE + iPRC FOR ONE kappa
# =============================================================================
 
class Cycle:
    pass
 
 
def build_cycle(kappa, tn=TN, n_periods=N_PERIODS, rtol=RTOL, atol=ATOL):
    """Fine limit cycle and iPRC, returned as interpolants of t."""
    p = dict(BASE, kappa=float(kappa))
 
    # --- past transients -----------------------------------------------------
    tr = solve_ivp(lambda t, y: f(t, y, p), (0, 5000), [0.49, 0.01, 0.01],
                   method='LSODA', rtol=rtol, atol=atol)
 
    # --- period from a Poincare section -------------------------------------
    # dI/dt = I*(beta*S - gamma) = 0 with I > 0  <=>  S = gamma/beta, and S
    # decreasing there is exactly the peak of I.  Event times are root-found on
    # the dense output, so T is not limited by an output grid.
    S_sec = p['gamma']/p['beta']
 
    def ev(t, y):
        return y[0] - S_sec
    ev.direction = -1.0
 
    sol_s = solve_ivp(lambda t, y: f(t, y, p), (0, 2000.0), tr.y[:, -1],
                      method='LSODA', rtol=rtol, atol=atol, events=ev)
    te = sol_s.t_events[0]
    if len(te) < 4:
        raise RuntimeError(f'too few section crossings at kappa={kappa}')
    T = float(np.mean(np.diff(te[-5:])))
 
    y_start = sol_s.y_events[0][-2]      # state exactly on the section
 
    # --- one cycle, finely sampled ------------------------------------------
    t_grid = np.linspace(0, T, tn)
    sol_c = solve_ivp(lambda t, y: f(t, y, p), (0, T), y_start,
                      method='LSODA', t_eval=t_grid, rtol=rtol, atol=atol)
    Sg, Ig, vg = sol_c.y
 
    S_i = make_periodic_spline(t_grid, Sg, T)
    I_i = make_periodic_spline(t_grid, Ig, T)
    v_i = make_periodic_spline(t_grid, vg, T)
 
    # --- adjoint, integrated backward ---------------------------------------
    def adj(t, Z):
        return -DF(S_i(t), I_i(t), v_i(t), p).T @ Z
 
    t_end = -n_periods*T
    t_eval = np.linspace(-(n_periods - 1)*T, t_end, tn)   # decreasing
    sol_a = solve_ivp(adj, (0.0, t_end), np.array([1.0, 1.0, 1.0]),
                      method='LSODA', t_eval=t_eval, rtol=rtol, atol=atol)
 
    Z_grid = sol_a.y[:, ::-1]            # reversing maps onto t = 0 .. T
 
    # --- normalise Z . gamma_dot = omega ------------------------------------
    # The product is exactly constant for the true solution, so rescale by a
    # single scalar; its spread is kept as a convergence diagnostic.
    gdot = np.array([f(0.0, sol_c.y[:, i], p) for i in range(tn)]).T
    prod = np.sum(Z_grid*gdot, axis=0)
    omega = 2*np.pi/T
    Z_grid = Z_grid*(omega/np.mean(prod))
    norm_spread = float(np.ptp(prod)/abs(np.mean(prod)))
    per_err = float(np.linalg.norm(Z_grid[:, -1] - Z_grid[:, 0]))
 
    Zv_i = make_periodic_spline(t_grid, Z_grid[2], T)
 
    def sigp_i(tq):
        """sigma'(xi(t)), built from the interpolated state."""
        xi = p['kappa']*(p['beta']*I_i(tq) - p['rho']
                         - p['theta']*(1 - 2*v_i(tq)))
        s = 1/(1 + np.exp(-xi))
        return s*(1 - s)
 
    c = Cycle()
    c.p, c.T, c.omega = p, T, omega
    c.v, c.Zv, c.sigp = v_i, Zv_i, sigp_i
    c.norm_spread, c.per_err = norm_spread, per_err
    return c
 
 
# =============================================================================
# H, C, AND THE DERIVATIVES AT pi
# =============================================================================
 
def H(phi, N, c):
    """Trapezoid rule on N subintervals of [0, T]. phi in radians."""
    t = np.linspace(0.0, c.T, N + 1)
    integrand = (c.Zv(t) * (-2) * c.p['kappa'] * c.p['theta'] * c.sigp(t)
                 * (c.v(t) - c.v(t + phi/c.omega)))
    return float(_trapz(integrand, t)/c.T)
 
 
def C(phi, N, c):
    return H(-phi, N, c) - H(phi, N, c)
 
 
def C_prime_pi(N, c, h=H_PHI):
    """Two-point centred difference.
 
        [C(pi+h) - C(pi-h)] / (2h)  =  C'(pi) + b h^2 + O(h^4)
    """
    return (C(np.pi + h, N, c) - C(np.pi - h, N, c))/(2*h)
 
 
def C_third_pi(N, c, h=H_PHI):
    """Four-point centred difference for the third derivative.
 
        [C(pi+2h) - 2C(pi+h) + 2C(pi-h) - C(pi-2h)] / (2h^3)
            =  C'''(pi) + (1/4) C^(5)(pi) h^2 + O(h^4)
    """
    return (C(np.pi + 2*h, N, c) - 2*C(np.pi + h, N, c)
            + 2*C(np.pi - h, N, c) - C(np.pi - 2*h, N, c))/(2*h**3)
 
 
# =============================================================================
# RUN
# =============================================================================
 
cyc_minus = build_cycle(KAPPA_STAR - DKAPPA)
cyc_star = build_cycle(KAPPA_STAR)
cyc_plus = build_cycle(KAPPA_STAR + DKAPPA)
 
print(f"kappa* = {KAPPA_STAR}   dkappa = {DKAPPA}   h_phi = {H_PHI}")
print(f"TN = {TN}   n_periods = {N_PERIODS}   rtol = atol = {RTOL}")
print(f"T(kappa*) = {cyc_star.T:.10f} days")
print(f"Z normalisation spread: {cyc_star.norm_spread:.3e}   "
      f"|Z(T)-Z(0)| = {cyc_star.per_err:.3e}")
 
# --- Table 1: convergence in the trapezoid mesh ------------------------------
print(f"\nTable 1: trapezoid mesh N        (h = {H_PHI})")
print(f"{'N':>6}  {'C_prime(pi,k*)':>16}  {'a':>16}  {'b = C3/6':>16}")
print("-"*60)
 
for N in N_LIST:
    cp_star = C_prime_pi(N, cyc_star)
    a = (C_prime_pi(N, cyc_plus) - C_prime_pi(N, cyc_minus))/(2*DKAPPA)
    b = C_third_pi(N, cyc_star)/6.0
    print(f"{N:6d}  {cp_star:+16.8e}  {a:+16.8e}  {b:+16.8e}")
 
# --- Table 2: sensitivity to the differentiation step ------------------------
# Large h: truncation (the b h^2 term) dominates.
# Small h: roundoff in the differenced C values dominates, fastest for C'''
#          because of the 1/h^3.
# The flat stretch in between is the usable window.
N_fine = N_LIST[-1]
print(f"\nTable 2: differentiation step h        (N = {N_fine})")
print(f"{'h':>9}  {'C_prime(pi,k*)':>16}  {'a':>16}  {'b = C3/6':>16}")
print("-"*62)
 
for h in H_SWEEP:
    cp_star = C_prime_pi(N_fine, cyc_star, h)
    a = (C_prime_pi(N_fine, cyc_plus, h)
         - C_prime_pi(N_fine, cyc_minus, h))/(2*DKAPPA)
    b = C_third_pi(N_fine, cyc_star, h)/6.0
    print(f"{h:9.5f}  {cp_star:+16.8e}  {a:+16.8e}  {b:+16.8e}")
 
print("\nNote: C'(pi, kappa*) is zero by construction, so its column is the")
print("residual bias rather than a measured quantity; a and b are the")
print("numbers of interest.")
print("\na < 0 and b > 0  =>  subcritical pitchfork")

kappa* = 387.0522   dkappa = 1.0   h_phi = 0.002
TN = 100000   n_periods = 20   rtol = atol = 1e-12
T(kappa*) = 91.1413469366 days
Z normalisation spread: 4.136e-10   |Z(T)-Z(0)| = 1.279e-08

Table 1: trapezoid mesh N        (h = 0.002)
     N    C_prime(pi,k*)                 a          b = C3/6
------------------------------------------------------------
   100   -4.17400295e-06   -8.30928949e-07   +7.10390051e-03
   200   -6.06182569e-08   -9.59099661e-07   +4.29498306e-04
   400   +1.47656193e-10   -9.57088781e-07   +3.14838578e-05
   800   +1.50511548e-10   -9.57088039e-07   +3.14116499e-05
  1600   +1.50525425e-10   -9.57088039e-07   +3.14097706e-05
  3200   +1.50521956e-10   -9.57088041e-07   +3.14103489e-05
  6400   +1.50525425e-10   -9.57088034e-07   +3.14102766e-05

Table 2: differentiation step h        (N = 6400)
        h    C_prime(pi,k*)                 a          b = C3/6
--------------------------------------------------------------
  0.50000   +1.06527724e-05   -1.228

## spline 

In [7]:
"""
C'(pi), b = C'''(pi)/6, and a = dC'(pi)/dkappa by SPLINE differentiation,
for spline orders k = 1 ... 5.

This is the "spline choice" leg of the convergence study.  The reference
column throughout is the plain centred finite difference at h = H_PHI, i.e.
exactly what ab_convergence_v3.py reports; the point of this script is to
check that an independent differentiation method lands on the same numbers.

  (A) local spline   -- interpolate C on a symmetric window [pi-W, pi+W] with
                        M points and differentiate the spline at pi.  W plays
                        the role of the finite-difference step h.
  (B) global spline  -- interpolate C over the whole of [0, 2*pi] on n_phi
                        points and differentiate at pi.  This is what the
                        original notebook did (UnivariateSpline, k=3, s=0),
                        so this table connects the published numbers to the
                        converged ones.
  (C) a              -- a is always a centred difference in kappa; what the
                        spline order controls is how C'(pi) is extracted at
                        each of the three kappa values.

A spline of order k has a piecewise-constant k-th derivative and vanishing
derivatives beyond that, so k = 1, 2 cannot produce C''' at all, and k = 3
gives a C''' that is piecewise constant and discontinuous exactly at the
knots -- pi is a knot here, so that entry is knot-placement dependent.
k = 4, 5 give a continuous third derivative and are the meaningful ones.
"""

import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline, UnivariateSpline

_trapz = getattr(np, 'trapezoid', None) or np.trapz

# =============================================================================
# CONFIGURATION
# =============================================================================

KAPPA_STAR = 387.0522
DKAPPA = 1.0           # centred-difference step in kappa, for a

H_PHI = 0.002          # centred-difference step in phi, for the reference
                       # column; same value as ab_convergence_v3.py

N_TRAP = 3200          # trapezoid mesh for H
TN = 100_000           # limit cycle resolution
N_PERIODS = 20
RTOL = ATOL = 1e-12

K_LIST = [1, 2, 3, 4, 5]                 # spline orders to test
W_LIST = [0.5, 0.25, 0.1, 0.05]          # half-widths of the local window
M_LOCAL = 21                             # points per local window (odd: pi is a node)
NPHI_LIST = [200, 400, 800]              # global grid sizes over [0, 2*pi]

BASE = dict(alpha=0.01, beta=0.5, gamma=1/7, eta=1/7, theta=0.01, rho=0.01)


# =============================================================================
# SYSTEM
# =============================================================================

def f(t, y, p):
    S, I, v = y
    return [p['alpha']*(1 - S - I) - p['beta']*S*I - p['eta']*v*S,
            -p['gamma']*I + p['beta']*S*I,
            -v + 1/(1 + np.exp(-p['kappa']*(p['beta']*I - p['rho']
                                            - p['theta']*(1 - 2*v))))]


_S, _I, _v = sp.symbols('S I v')
_al, _be, _ga, _et, _th, _ka, _rh = sp.symbols(
    'alpha beta gamma eta theta kappa rho')
_F = sp.Matrix([
    _al*(1 - _S - _I) - _be*_S*_I - _et*_v*_S,
    -_ga*_I + _be*_S*_I,
    -_v + 1/(1 + sp.exp(-_ka*(_be*_I - _rh - _th*(1 - 2*_v))))])
_DF_func = sp.lambdify((_S, _I, _v, _al, _be, _ga, _et, _th, _ka, _rh),
                       _F.jacobian(sp.Matrix([_S, _I, _v])), modules='numpy')


def DF(S, I, v, p):
    return np.array(_DF_func(S, I, v, p['alpha'], p['beta'], p['gamma'],
                             p['eta'], p['theta'], p['kappa'], p['rho']))


def make_periodic_spline(t, y, T):
    yy = np.asarray(y, dtype=float).copy()
    yy[-1] = yy[0]
    cs = CubicSpline(t, yy, bc_type='periodic')
    return lambda tq: cs(np.mod(tq, T))


class Cycle:
    pass


def build_cycle(kappa, tn=TN, n_periods=N_PERIODS, rtol=RTOL, atol=ATOL):
    p = dict(BASE, kappa=float(kappa))

    tr = solve_ivp(lambda t, y: f(t, y, p), (0, 5000), [0.49, 0.01, 0.01],
                   method='LSODA', rtol=rtol, atol=atol)

    S_sec = p['gamma']/p['beta']

    def ev(t, y):
        return y[0] - S_sec
    ev.direction = -1.0

    sol_s = solve_ivp(lambda t, y: f(t, y, p), (0, 2000.0), tr.y[:, -1],
                      method='LSODA', rtol=rtol, atol=atol, events=ev)
    te = sol_s.t_events[0]
    T = float(np.mean(np.diff(te[-5:])))
    y_start = sol_s.y_events[0][-2]

    t_grid = np.linspace(0, T, tn)
    sol_c = solve_ivp(lambda t, y: f(t, y, p), (0, T), y_start,
                      method='LSODA', t_eval=t_grid, rtol=rtol, atol=atol)
    Sg, Ig, vg = sol_c.y
    S_i = make_periodic_spline(t_grid, Sg, T)
    I_i = make_periodic_spline(t_grid, Ig, T)
    v_i = make_periodic_spline(t_grid, vg, T)

    def adj(t, Z):
        return -DF(S_i(t), I_i(t), v_i(t), p).T @ Z

    t_end = -n_periods*T
    sol_a = solve_ivp(adj, (0.0, t_end), np.array([1.0, 1.0, 1.0]),
                      method='LSODA', rtol=rtol, atol=atol,
                      t_eval=np.linspace(-(n_periods - 1)*T, t_end, tn))
    Z_grid = sol_a.y[:, ::-1]

    gdot = np.array([f(0.0, sol_c.y[:, i], p) for i in range(tn)]).T
    omega = 2*np.pi/T
    Z_grid = Z_grid*(omega/np.mean(np.sum(Z_grid*gdot, axis=0)))

    c = Cycle()
    c.p, c.T, c.omega = p, T, omega
    c.v, c.Zv = v_i, make_periodic_spline(t_grid, Z_grid[2], T)

    def sigp_i(tq):
        xi = p['kappa']*(p['beta']*I_i(tq) - p['rho']
                         - p['theta']*(1 - 2*v_i(tq)))
        s = 1/(1 + np.exp(-xi))
        return s*(1 - s)
    c.sigp = sigp_i
    return c


# =============================================================================
# H, C
# =============================================================================

def H(phi, N, c):
    t = np.linspace(0.0, c.T, N + 1)
    integrand = (c.Zv(t) * (-2) * c.p['kappa'] * c.p['theta'] * c.sigp(t)
                 * (c.v(t) - c.v(t + phi/c.omega)))
    return float(_trapz(integrand, t)/c.T)


def C(phi, N, c):
    return H(-phi, N, c) - H(phi, N, c)


def C_vec(phis, N, c):
    return np.array([C(x, N, c) for x in phis])


# --- reference: plain centred finite differences ----------------------------

def C_prime_pi(N, c, h=H_PHI):
    """[C(pi+h) - C(pi-h)] / (2h)."""
    return (C(np.pi + h, N, c) - C(np.pi - h, N, c))/(2*h)


def C_third_pi(N, c, h=H_PHI):
    """[C(pi+2h) - 2C(pi+h) + 2C(pi-h) - C(pi-2h)] / (2h^3)."""
    return (C(np.pi + 2*h, N, c) - 2*C(np.pi + h, N, c)
            + 2*C(np.pi - h, N, c) - C(np.pi - 2*h, N, c))/(2*h**3)


# --- spline differentiation --------------------------------------------------

def spline_derivs(phis, Cvals, k):
    """Interpolating spline of order k; return C'(pi) and C'''(pi).

    UnivariateSpline with s=0 interpolates. Derivatives of order > k are
    identically zero, so C''' is unavailable for k < 3.
    """
    spl = UnivariateSpline(phis, Cvals, k=k, s=0)
    d1 = float(spl.derivative(1)(np.pi))
    d3 = float(spl.derivative(3)(np.pi)) if k >= 3 else np.nan
    return d1, d3


# =============================================================================
# RUN
# =============================================================================

cyc = build_cycle(KAPPA_STAR)

cp_ref = C_prime_pi(N_TRAP, cyc)
b_ref = C_third_pi(N_TRAP, cyc)/6.0

print(f"kappa* = {KAPPA_STAR}   T = {cyc.T:.10f} days   N_trap = {N_TRAP}")
print(f"\nREFERENCE (centred finite difference, h = {H_PHI})")
print(f"   C'(pi) = {cp_ref:+.8e}")
print(f"   b      = {b_ref:+.8e}")

# --- (A) local spline on [pi-W, pi+W] ---------------------------------------
print(f"\n(A) LOCAL spline, {M_LOCAL} points on [pi-W, pi+W]")
print(f"{'W':>8} {'spacing':>10} {'k':>3} {'C_prime(pi)':>17} {'b = C3/6':>17}")
print('-'*60)

for W in W_LIST:
    phis = np.linspace(np.pi - W, np.pi + W, M_LOCAL)
    Cv = C_vec(phis, N_TRAP, cyc)
    for k in K_LIST:
        d1, d3 = spline_derivs(phis, Cv, k)
        b = d3/6.0 if np.isfinite(d3) else np.nan
        bs = f'{b:+17.8e}' if np.isfinite(b) else f"{'n/a (k<3)':>17}"
        print(f'{W:8.3f} {2*W/(M_LOCAL-1):10.5f} {k:3d} {d1:+17.8e} {bs}')
    print()

# --- (B) global spline on [0, 2*pi], as the original notebook did ------------
print(f"(B) GLOBAL spline on [0, 2*pi]   (the notebook's approach)")
print(f"{'n_phi':>8} {'spacing':>10} {'k':>3} {'C_prime(pi)':>17} {'b = C3/6':>17}")
print('-'*60)

for nphi in NPHI_LIST:
    phis = np.linspace(0.0, 2*np.pi, nphi)
    Cv = C_vec(phis, N_TRAP, cyc)
    for k in K_LIST:
        d1, d3 = spline_derivs(phis, Cv, k)
        b = d3/6.0 if np.isfinite(d3) else np.nan
        bs = f'{b:+17.8e}' if np.isfinite(b) else f"{'n/a (k<3)':>17}"
        print(f'{nphi:8d} {2*np.pi/(nphi-1):10.5f} {k:3d} {d1:+17.8e} {bs}')
    print()

# --- (C) a = dC'(pi)/dkappa, with C'(pi) taken from the spline -------------
cyc_m = build_cycle(KAPPA_STAR - DKAPPA)
cyc_p = build_cycle(KAPPA_STAR + DKAPPA)

a_ref = (C_prime_pi(N_TRAP, cyc_p) - C_prime_pi(N_TRAP, cyc_m))/(2*DKAPPA)

print(f"(C) a = [C'(pi,k*+dk) - C'(pi,k*-dk)] / (2 dk),  dk = {DKAPPA}")
print(f"    C'(pi) at each kappa from the LOCAL spline")
print(f"{'W':>8} {'spacing':>10} {'k':>3} {'a':>17}")
print('-'*42)

for W in W_LIST:
    ph = np.linspace(np.pi - W, np.pi + W, M_LOCAL)
    Cm, Cp = C_vec(ph, N_TRAP, cyc_m), C_vec(ph, N_TRAP, cyc_p)
    for k in K_LIST:
        d1m, _ = spline_derivs(ph, Cm, k)
        d1p, _ = spline_derivs(ph, Cp, k)
        print(f'{W:8.3f} {2*W/(M_LOCAL-1):10.5f} {k:3d} '
              f'{(d1p - d1m)/(2*DKAPPA):+17.8e}')
    print()

print(f"reference (centred FD, h = {H_PHI}):  C'(pi) = {cp_ref:+.8e}   "
      f"b = {b_ref:+.8e}   a = {a_ref:+.8e}")

kappa* = 387.0522   T = 91.1413469366 days   N_trap = 3200

REFERENCE (centred finite difference, h = 0.002)
   C'(pi) = +1.50521956e-10
   b      = +3.14103489e-05

(A) LOCAL spline, 21 points on [pi-W, pi+W]
       W    spacing   k       C_prime(pi)          b = C3/6
------------------------------------------------------------
   0.500    0.05000   1   +7.87876937e-08         n/a (k<3)
   0.500    0.05000   2   +1.94473863e-08         n/a (k<3)
   0.500    0.05000   3   -1.33534894e-10   +3.15684914e-05
   0.500    0.05000   4   -9.93044383e-12   +3.14889891e-05
   0.500    0.05000   5   +2.46930755e-11   +3.14092343e-05

   0.250    0.02500   1   +1.96707259e-08         n/a (k<3)
   0.250    0.02500   2   +4.91956114e-09         n/a (k<3)
   0.250    0.02500   3   +1.49433717e-11   +3.14492520e-05
   0.250    0.02500   4   +2.27088054e-11   +3.14293516e-05
   0.250    0.02500   5   +2.48907720e-11   +3.14094241e-05

   0.100    0.01000   1   +3.16621382e-09         n/a (k<3)
   0.10

## next we sweep through rtol atol 

In [9]:
"""
Integration-tolerance leg of the convergence study.

Everything below build_cycle is identical to ab_convergence_v3.py.  The only
new part is the RUN section, which rebuilds the limit cycle and iPRC under
different solver settings and reports C'(pi) and b each time.

build_cycle passes rtol/atol to ALL FOUR solve_ivp calls:
    1. transient        (0 -> 5000 days)
    2. period finding   (0 -> 2000 days, event root-find for T)
    3. one cycle        (0 -> T, produces Gamma)
    4. adjoint          (0 -> -n_periods*T, produces Z)
so one sweep of the pair covers every instance.

Three sweeps, each varying ONE setting with the others held at the base
values.  Every row rebuilds one cycle from scratch at kappa*; a and the
kappa-difference are not recomputed here (that needs three cycles per row
and a is already known to be robust).

Two columns are internal consistency checks that need no reference value:
    Z.gdot spread -- Z . gamma_dot is mathematically constant along the
                     cycle, so its relative variation measures convergence
    |Z(T)-Z(0)|   -- Z must be T-periodic, so this is the closure error
"""

import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline

_trapz = getattr(np, 'trapezoid', None) or np.trapz   # numpy 1.x / 2.x

# =============================================================================
# CONFIGURATION
# =============================================================================

KAPPA_STAR = 387.0522
H_PHI  = 0.002        # FD step in phi (same as ab_convergence_v3.py)
N_TRAP = 3200         # trapezoid mesh for H (converged at this h)

# --- base settings: each sweep varies one of these, holding the others ------
TN_BASE    = 20_000   # reduced from 100_000 purely for speed; C'(pi) was
                      # shown stable to 7 digits over TN = 2000 .. 100_000
NPER_BASE  = 20
TOL_BASE   = 1e-12

# --- the three sweeps --------------------------------------------------------
TOL_LIST  = [1e-8, 1e-9, 1e-10, 1e-11, 1e-12, 1e-13]
TN_LIST   = [5_000, 10_000, 20_000, 50_000]
NPER_LIST = [5, 10, 20, 40]

BASE = dict(alpha=0.01, beta=0.5, gamma=1/7, eta=1/7, theta=0.01, rho=0.01)


# =============================================================================
# SYSTEM
# =============================================================================

def f(t, y, p):
    S, I, v = y
    return [p['alpha']*(1 - S - I) - p['beta']*S*I - p['eta']*v*S,
            -p['gamma']*I + p['beta']*S*I,
            -v + 1/(1 + np.exp(-p['kappa']*(p['beta']*I - p['rho']
                                            - p['theta']*(1 - 2*v))))]


_S, _I, _v = sp.symbols('S I v')
_al, _be, _ga, _et, _th, _ka, _rh = sp.symbols(
    'alpha beta gamma eta theta kappa rho')

_F = sp.Matrix([
    _al*(1 - _S - _I) - _be*_S*_I - _et*_v*_S,
    -_ga*_I + _be*_S*_I,
    -_v + 1/(1 + sp.exp(-_ka*(_be*_I - _rh - _th*(1 - 2*_v))))])

_DF_func = sp.lambdify((_S, _I, _v, _al, _be, _ga, _et, _th, _ka, _rh),
                       _F.jacobian(sp.Matrix([_S, _I, _v])), modules='numpy')


def DF(S, I, v, p):
    return np.array(_DF_func(S, I, v, p['alpha'], p['beta'], p['gamma'],
                             p['eta'], p['theta'], p['kappa'], p['rho']))


def make_periodic_spline(t, y, T):
    yy = np.asarray(y, dtype=float).copy()
    yy[-1] = yy[0]
    cs = CubicSpline(t, yy, bc_type='periodic')
    return lambda tq: cs(np.mod(tq, T))


# =============================================================================
# LIMIT CYCLE + iPRC  (identical to ab_convergence_v3.py)
# =============================================================================

class Cycle:
    pass


def build_cycle(kappa, tn=TN_BASE, n_periods=NPER_BASE,
                rtol=TOL_BASE, atol=TOL_BASE):
    p = dict(BASE, kappa=float(kappa))

    # (1) transient
    tr = solve_ivp(lambda t, y: f(t, y, p), (0, 5000), [0.49, 0.01, 0.01],
                   method='LSODA', rtol=rtol, atol=atol)

    # (2) period from a Poincare section
    S_sec = p['gamma']/p['beta']

    def ev(t, y):
        return y[0] - S_sec
    ev.direction = -1.0

    sol_s = solve_ivp(lambda t, y: f(t, y, p), (0, 2000.0), tr.y[:, -1],
                      method='LSODA', rtol=rtol, atol=atol, events=ev)
    te = sol_s.t_events[0]
    if len(te) < 4:
        raise RuntimeError(f'too few section crossings at kappa={kappa}')
    T = float(np.mean(np.diff(te[-5:])))
    y_start = sol_s.y_events[0][-2]

    # (3) one cycle
    t_grid = np.linspace(0, T, tn)
    sol_c = solve_ivp(lambda t, y: f(t, y, p), (0, T), y_start,
                      method='LSODA', t_eval=t_grid, rtol=rtol, atol=atol)
    Sg, Ig, vg = sol_c.y

    S_i = make_periodic_spline(t_grid, Sg, T)
    I_i = make_periodic_spline(t_grid, Ig, T)
    v_i = make_periodic_spline(t_grid, vg, T)

    # (4) adjoint, integrated backward
    def adj(t, Z):
        return -DF(S_i(t), I_i(t), v_i(t), p).T @ Z

    t_end = -n_periods*T
    t_eval = np.linspace(-(n_periods - 1)*T, t_end, tn)
    sol_a = solve_ivp(adj, (0.0, t_end), np.array([1.0, 1.0, 1.0]),
                      method='LSODA', t_eval=t_eval, rtol=rtol, atol=atol)
    Z_grid = sol_a.y[:, ::-1]

    gdot = np.array([f(0.0, sol_c.y[:, i], p) for i in range(tn)]).T
    prod = np.sum(Z_grid*gdot, axis=0)
    omega = 2*np.pi/T
    Z_grid = Z_grid*(omega/np.mean(prod))

    c = Cycle()
    c.p, c.T, c.omega = p, T, omega
    c.v = v_i
    c.Zv = make_periodic_spline(t_grid, Z_grid[2], T)
    c.norm_spread = float(np.ptp(prod)/abs(np.mean(prod)))
    c.per_err = float(np.linalg.norm(Z_grid[:, -1] - Z_grid[:, 0]))

    def sigp_i(tq):
        xi = p['kappa']*(p['beta']*I_i(tq) - p['rho']
                         - p['theta']*(1 - 2*v_i(tq)))
        s = 1/(1 + np.exp(-xi))
        return s*(1 - s)
    c.sigp = sigp_i
    return c


# =============================================================================
# H, C, DERIVATIVES  (identical to ab_convergence_v3.py)
# =============================================================================

def H(phi, N, c):
    t = np.linspace(0.0, c.T, N + 1)
    integrand = (c.Zv(t) * (-2) * c.p['kappa'] * c.p['theta'] * c.sigp(t)
                 * (c.v(t) - c.v(t + phi/c.omega)))
    return float(_trapz(integrand, t)/c.T)


def C(phi, N, c):
    return H(-phi, N, c) - H(phi, N, c)


def C_prime_pi(N, c, h=H_PHI):
    return (C(np.pi + h, N, c) - C(np.pi - h, N, c))/(2*h)


def C_third_pi(N, c, h=H_PHI):
    return (C(np.pi + 2*h, N, c) - 2*C(np.pi + h, N, c)
            + 2*C(np.pi - h, N, c) - C(np.pi - 2*h, N, c))/(2*h**3)


# =============================================================================
# RUN
# =============================================================================

HEAD = (f"{'setting':>14} {'T':>15} {'C_prime(pi)':>15} {'b = C3/6':>16} "
        f"{'Z.gdot spread':>14} {'|Z(T)-Z(0)|':>13}")


def row(label, **kw):
    """Build one cycle with the given override and print one line."""
    try:
        c = build_cycle(KAPPA_STAR, **kw)
    except Exception as e:
        print(f'{label:>14}   FAILED: {e}')
        return
    print(f'{label:>14} {c.T:15.9f} {C_prime_pi(N_TRAP, c):+15.5e} '
          f'{C_third_pi(N_TRAP, c)/6:+16.8e} '
          f'{c.norm_spread:14.2e} {c.per_err:13.2e}', flush=True)


print(f"kappa* = {KAPPA_STAR}   h_phi = {H_PHI}   N_trap = {N_TRAP}")
print(f"base settings: TN = {TN_BASE}, n_periods = {NPER_BASE}, "
      f"rtol = atol = {TOL_BASE}")

print(f"\n(1) SOLVER TOLERANCE   rtol = atol, applied to all four solve_ivp calls")
print(HEAD)
print('-'*92)
for tol in TOL_LIST:
    row(f'{tol:.0e}', rtol=tol, atol=tol)

print(f"\n(2) LIMIT CYCLE RESOLUTION TN   (spline knot density for Gamma and Z)")
print(HEAD)
print('-'*92)
for tn in TN_LIST:
    row(f'{tn}', tn=tn)

print(f"\n(3) ADJOINT PERIODS   (how long Z is integrated before the last cycle)")
print(HEAD)
print('-'*92)
for npd in NPER_LIST:
    row(f'{npd}', n_periods=npd)

kappa* = 387.0522   h_phi = 0.002   N_trap = 3200
base settings: TN = 20000, n_periods = 20, rtol = atol = 1e-12

(1) SOLVER TOLERANCE   rtol = atol, applied to all four solve_ivp calls
       setting               T     C_prime(pi)         b = C3/6  Z.gdot spread   |Z(T)-Z(0)|
--------------------------------------------------------------------------------------------
         1e-08    91.141233029    -1.64954e-09  +3.14248049e-05       3.07e-06      6.76e-05
         1e-09    91.141345472    -3.08018e-11  +3.14103489e-05       1.86e-07      2.43e-06
         1e-10    91.141346939    +1.14287e-10  +3.14112162e-05       6.58e-08      7.83e-07
         1e-11    91.141346835    +1.42806e-10  +3.14099152e-05       4.14e-09      1.53e-07
         1e-12    91.141346937    +1.50553e-10  +3.14103489e-05       4.54e-10      1.01e-08
         1e-13    91.141346959    +1.51743e-10  +3.14102043e-05       3.66e-11      1.03e-09

(2) LIMIT CYCLE RESOLUTION TN   (spline knot density for Gamma and Z)